# DESI∞ INFINITY SYSTEM: AI Physics Engine
### Phase 3: Neural Network Approximation of Thermodynamic Heat Transfer

**Author:** Franklin Obunezi
**Project Goal:** To build the computational AI foundation for the DESI∞ bio-computer, starting with the ability to simulate physical heat transfer for medical cooling systems.

---

#### 1. Introduction
The DESI∞ system requires precise thermal management. To test the feasibility of using machine learning to predict thermal dynamics, I built a Neural Network designed to learn two core laws of physics:
1. **Conduction:** Q = k * A * (dT / L)
2. **Convection:** Q = h * A * dT

This notebook generates the physics data, trains the model, and visually verifies the AI's accuracy against the analytic equations.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)
num_samples = 5000

# 1. Conduction Data
k = np.random.uniform(0.1, 400.0, num_samples)
A_c = np.random.uniform(0.01, 1.0, num_samples)
dT_c = np.random.uniform(1.0, 500.0, num_samples)
L = np.random.uniform(0.05, 0.5, num_samples)
Q_conduction = (k * A_c * dT_c) / L

# 2. Convection Data
h = np.random.uniform(5.0, 5000.0, num_samples)
A_v = np.random.uniform(0.01, 5.0, num_samples)
dT_v = np.random.uniform(1.0, 300.0, num_samples)
Q_convection = h * A_v * dT_v

print(f"Generated {num_samples} samples for Conduction and Convection.")
print(f"Sample Conduction Q (W): {Q_conduction[0]:.2f}")
print(f"Sample Convection Q (W): {Q_convection[0]:.2f}")

#### 2. Neural Network Architecture
I will use a Multi-Layer Perceptron (MLP) built in PyTorch. Because the input variables for Conduction (4 inputs) and Convection (3 inputs) are different, I will train a model to predict the **heat transfer coefficient multiplier** to unify the physics, or train them separately.

For this baseline report, we will train the AI on the **Conduction** dataset to prove the pipeline works.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Prepare Conduction Data
X = np.column_stack((k, A_c, dT_c, L))
y = Q_conduction.reshape(-1, 1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.10, random_state=42)

scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_train_s = scaler_X.fit_transform(X_train)
X_test_s = scaler_X.transform(X_test)
y_train_s = scaler_y.fit_transform(y_train)
y_test_s = scaler_y.transform(y_test)

X_train_t = torch.FloatTensor(X_train_s)
y_train_t = torch.FloatTensor(y_train_s)
X_test_t = torch.FloatTensor(X_test_s)

model = nn.Sequential(
    nn.Linear(4, 64),
    nn.ReLU(),
    nn.Linear(64, 64),
    nn.ReLU(),
    nn.Linear(64, 1)
)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

epochs = 500
for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()

model.eval()
with torch.no_grad():
    preds_s = model(X_test_t)
    preds = scaler_y.inverse_transform(preds_s.numpy())

mae = np.mean(np.abs(preds - y_test))
max_error = np.max((np.abs(preds - y_test) / y_test) * 100)

print(f"Training Complete.")
print(f"Max Relative Error: {max_error:.4f}%")
print(f"Mean Absolute Error: {mae:.4f} W")

#### 3. Visual Verification
The true test of a physics-informed AI is visual. Below is the scatter plot comparing the AI's predictions against the exact analytic equation.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
plt.scatter(y_test, preds, alpha=0.3, color='blue', label='AI Predictions')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Accuracy')
plt.xlabel('Actual Heat Transfer (Watts)')
plt.ylabel('AI Predicted Heat Transfer (Watts)')
plt.title('DESI∞ AI Verification: Conduction Physics')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()